In [ ]:
import numpy as np, pandas as pd
from scipy.stats import loguniform
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor, StackingRegressor
from sklearn.feature_selection import SelectKBest, VarianceThreshold, f_regression
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel
from sklearn.impute import KNNImputer
from sklearn.linear_model import RidgeCV
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR


class MADOutlierMasker(BaseEstimator, TransformerMixin):
    """Set entries more than `thresh` robust sigmas from the column median to NaN."""
    def __init__(self, thresh=8.0):
        self.thresh = thresh

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        self.med_ = np.nanmedian(X, axis=0)
        mad = np.nanmedian(np.abs(X - self.med_), axis=0)
        q1, q3 = np.nanpercentile(X, [25, 75], axis=0)
        scale = np.where(mad > 0, mad / 0.6745, (q3 - q1) / 1.349)
        self.scale_ = np.where(scale > 0, scale, np.inf)
        return self

    def transform(self, X):
        X = np.array(X, dtype=float)
        X[np.abs(X - self.med_) / self.scale_ > self.thresh] = np.nan
        return X


def make(model, k=100, thresh=8.0, n_neighbors=5):
    return Pipeline([
        ('mask', MADOutlierMasker(thresh)),
        ('imp', KNNImputer(n_neighbors=n_neighbors)),
        ('var', VarianceThreshold(0.0)),
        ('scale', StandardScaler()),
        ('sel', SelectKBest(f_regression, k=k)),
        ('model', model),
    ])

In [ ]:
Xtr = pd.read_csv('X_train.csv')
ytr = pd.read_csv('y_train.csv')
Xte = pd.read_csv('X_test.csv')
X = Xtr.drop(columns='id')
y = ytr['y'].to_numpy()

inner_cv = KFold(n_splits=3, shuffle=True, random_state=0)
outer_cv = KFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
The following evaluation tunes each model only within the outer training fold. This gives held-out predictions for both the individual models and the stack, so the residual-correlation check and reported R² do not reuse each fold's target values during tuning.

gpr_kernels = [
    ConstantKernel(1.0, (1e-2, 1e2))
    * Matern(length_scale=1.0, length_scale_bounds=(1e-2, 1e2), nu=nu)
    + WhiteKernel(noise_level=1e-2, noise_level_bounds=(1e-5, 1e1))
    for nu in (1.5, 2.5)
]

gpr = GaussianProcessRegressor(
    kernel=gpr_kernels[0], alpha=1e-6, normalize_y=True,
    n_restarts_optimizer=0, random_state=42,
)

model_specs = {
    'svr': {
        'estimator': make(TransformedTargetRegressor(SVR(), transformer=StandardScaler())),
        'params': {
            'sel__k': [50, 100, 150, 200],
            'mask__thresh': [np.inf, 8, 12],
            'model__regressor__C': loguniform(0.3, 300),
            'model__regressor__gamma': loguniform(1e-4, 5e-2),
            'model__regressor__epsilon': [0.03, 0.05, 0.1, 0.2],
        },
        'n_iter': 8,
    },
    'gpr': {
        'estimator': make(gpr, k=40, thresh=np.inf),
        'params': {
            'sel__k': [20, 40, 60],
            'model__kernel': gpr_kernels,
            'model__alpha': [1e-8, 1e-6, 1e-4],
        },
        'n_iter': 6,
    },
    'extra_trees': {
        'estimator': make(ExtraTreesRegressor(random_state=42, n_jobs=1)),
        'params': {
            'sel__k': [50, 100, 150],
            'mask__thresh': [np.inf, 8],
            'model__n_estimators': [300, 600],
            'model__max_features': [0.4, 0.7, 1.0],
            'model__min_samples_leaf': [2, 5, 10],
            'model__max_depth': [None, 15, 30],
        },
        'n_iter': 6,
    },
    'hist_gb': {
        'estimator': make(HistGradientBoostingRegressor(random_state=42)),
        'params': {
            'sel__k': [50, 100, 150],
            'mask__thresh': [np.inf, 8],
            'model__max_iter': [300, 600],
            'model__learning_rate': [0.03, 0.06, 0.1],
            'model__max_leaf_nodes': [7, 15, 31],
            'model__min_samples_leaf': [20, 40],
            'model__l2_regularization': [0.1, 1.0, 5.0],
        },
        'n_iter': 6,
    },
}

In [ ]:
model_names = list(model_specs)
oof = {name: np.full(len(y), np.nan) for name in [*model_names, 'stack']}
fold_scores = {name: [] for name in [*model_names, 'stack']}
fold_params = {name: [] for name in model_names}

for fold, (train_idx, valid_idx) in enumerate(outer_cv.split(X, y), start=1):
    X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
    y_train, y_valid = y[train_idx], y[valid_idx]
    fold_searches = {}

    for name, spec in model_specs.items():
        search = RandomizedSearchCV(
            estimator=spec['estimator'],
            param_distributions=spec['params'],
            n_iter=spec['n_iter'],
            scoring='r2',
            cv=inner_cv,
            n_jobs=-1,
            random_state=100 + fold,
            error_score='raise',
        )
        search.fit(X_train, y_train)
        pred = search.predict(X_valid)
        oof[name][valid_idx] = pred
        fold_scores[name].append(r2_score(y_valid, pred))
        fold_params[name].append(search.best_params_)
        fold_searches[name] = search
        print(f'fold={fold} {name}: R2={fold_scores[name][-1]:.4f}')

    stack = StackingRegressor(
        estimators=[(name, search.best_estimator_) for name, search in fold_searches.items()],
        final_estimator=RidgeCV(alphas=np.logspace(-3, 3, 13)),
        cv=KFold(n_splits=3, shuffle=True, random_state=fold),
        n_jobs=1,
    )
    stack.fit(X_train, y_train)
    stack_pred = stack.predict(X_valid)
    oof['stack'][valid_idx] = stack_pred
    fold_scores['stack'].append(r2_score(y_valid, stack_pred))
    print(f'fold={fold} stack: R2={fold_scores["stack"][-1]:.4f}')

performance = pd.DataFrame([
    {'model': name, 'mean_r2': np.mean(scores), 'std_r2': np.std(scores, ddof=1)}
    for name, scores in fold_scores.items()
]).sort_values('mean_r2', ascending=False).set_index('model')
print('Nested 5-fold outer-CV results:')
print(performance.round(4))

residuals = pd.DataFrame({name: oof[name] - y for name in oof}, index=X.index)
residual_corr = residuals.corr()
print('Out-of-fold residual correlations:')
print(residual_corr.round(2))

params_table = pd.DataFrame([
    {'model': name, 'fold': fold, 'best_params': params}
    for name, folds in fold_params.items()
    for fold, params in enumerate(folds, start=1)
])
print('Best parameters selected inside each outer training fold:')
print(params_table.to_string(index=False))

In [ ]:
Interpret residual correlations together with the outer-CV scores: high correlation means two models tend to make similar mistakes, while lower correlation suggests more complementary predictions. The stack is scored on the same held-out folds as the individual models. The search budgets below are intentionally small for an initial run; increase each `n_iter` after confirming runtime.

# Refit each model's randomized search on all training data for the final stack.
# These inner-CV scores are for tuning only; use the nested outer-CV table above for performance.
final_searches = {}
for name, spec in model_specs.items():
    search = RandomizedSearchCV(
        estimator=spec['estimator'],
        param_distributions=spec['params'],
        n_iter=spec['n_iter'],
        scoring='r2',
        cv=inner_cv,
        n_jobs=-1,
        random_state=2026,
        error_score='raise',
    )
    search.fit(X, y)
    final_searches[name] = search
    print(f'{name}: inner-CV R2={search.best_score_:.4f}')
    print(f'  best params: {search.best_params_}')

In [ ]:
pairs = [
    (a, b, residual_corr.loc[a, b])
    for i, a in enumerate(model_names)
    for b in model_names[i + 1:]
]
pairs.sort(key=lambda item: abs(item[2]), reverse=True)
pairwise_residual_corr = pd.DataFrame(pairs, columns=['model_a', 'model_b', 'residual_corr'])
print('Base-model residual correlation pairs, highest absolute correlation first:')
print(pairwise_residual_corr.round(3).to_string(index=False))

In [ ]:
models2 = {
    name: search.best_estimator_
    for name, search in final_searches.items()
}
stack = StackingRegressor(
    estimators=list(models2.items()),
    final_estimator=RidgeCV(alphas=np.logspace(-3, 3, 13)),
    cv=KFold(n_splits=5, shuffle=True, random_state=0),
    n_jobs=1,
)
print('Final stack configured with:', ', '.join(models2))

In [ ]:
stack.fit(X, y)
Xt = Xte.drop(columns='id')
pred = np.clip(stack.predict(Xt), y.min(), y.max())

sub = pd.read_csv('sample.csv')
assert (sub['id'] == Xte['id']).all(), 'id mismatch with sample.csv'
sub['y'] = pred
sub.to_csv('submission.csv', index=False)
print(f'Wrote submission.csv ({len(sub)} rows)')

In [62]:
stack.fit(X, y)
Xt   = Xte.drop(columns='id')
pred = np.clip(stack.predict(Xt), y.min(), y.max())

sub = pd.read_csv('sample.csv')
assert (sub['id'] == Xte['id']).all()
sub['y'] = pred
sub.to_csv('submission.csv', index=False)